# NIRF Level 1 · Four submission questions (2.3, 2.4, 5.3, 7.1)

Interactive Plotly versions of the charts in the Streamlit dashboard, built on `nirf_2017_2025_unified.csv`.

**How to run:** place this notebook in `Notebooks/`, make sure `data/processed/nirf_2017_2025_unified.csv` exists, then *Run All*. Each question saves an HTML file (interactive) and a PNG (for screenshots) into `Screenshots/`. Take the PNG for the submission.

**Requirements:** `pip install plotly kaleido==0.2.1 pandas numpy`
(`kaleido==0.2.1` is needed for PNG export. If PNG export fails, open the HTML file in a browser and screenshot it.)

## 1. Setup and configuration

In [27]:
import json
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def find_project_root():
    """Walk up from the notebook folder until the folder containing data/ is found."""
    here = Path.cwd()
    for folder in [here, *here.parents]:
        if (folder / "data").exists():
            return folder
    return here


ROOT = find_project_root()
DATA_PATH = ROOT / "data" / "processed" / "nirf_2017_2025_unified.csv"
OUT_DIR = ROOT / "Screenshots"
TABLE_DIR = OUT_DIR / "tables"
GEOJSON_PATH = ROOT / "data" / "reference" / "india_states.geojson"
GEOJSON_URLS = [
    "https://raw.githubusercontent.com/geohacker/india/master/state/india_state.geojson",
]

CATEGORY = "Overall"
TOP_N = 100
START_YEAR, END_YEAR = 2017, 2025
COMPARE_STATES = ["Karnataka", "Tamil Nadu"]
TLR_PR_YEAR = 2025

# Okabe-Ito palette: colour-blind safe, so blue vs orange/vermillion stays readable
PAL = {
    "gain": "#0072B2",     # blue
    "loss": "#D55E00",     # vermillion
    "neutral": "#9a9a9a",  # grey
    "state_a": "#0072B2",
    "state_b": "#E69F00",  # orange
}

# Diverging scale for change: vermillion (lost) -> dark grey (no change) -> blue (gained)
CHANGE_SCALE = [[0.0, "#D55E00"], [0.5, "#3b3b3b"], [1.0, "#0072B2"]]
# Sequential scale for counts (light = more institutions)
COUNT_SCALE = "Blues"
TEMPLATE = "plotly_dark"

print("Project root:", ROOT)

Project root: /Users/anubhav/Downloads/DHV_PROJECT/DHV_Final_Project


## 2. Load and prepare the unified dataset

In [28]:
COLUMN_MAP = {
    "Year": "year", "Category": "category", "Unified_ID": "institution_id",
    "Name_Unified": "institution", "Name_Display": "display_name",
    "State_Unified": "state", "Rank": "rank", "Score": "score",
    "TLR": "TLR", "RPC": "RPC", "GO": "GO", "OI": "OI", "Perception": "PR",
}


def load_nirf(path=DATA_PATH):
    df = pd.read_csv(path).rename(columns=COLUMN_MAP)
    df["year"] = df["year"].astype(int)
    df["institution_id"] = df["institution_id"].astype(str)   # join key: never the name
    for col in ["rank", "score", "TLR", "RPC", "GO", "OI", "PR"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    before = len(df)
    df = df.drop_duplicates()
    dup = df.duplicated(["year", "category", "institution_id"], keep=False)
    if dup.any():
        raise ValueError(f"{dup.sum()} rows repeat an institution within a year and category")
    print(f"Loaded {len(df):,} rows ({before - len(df)} exact duplicates removed)")
    return df


df = load_nirf()
overall = df[df["category"] == CATEGORY].copy()
ALL_STATES = sorted(overall["state"].dropna().unique())
print(f"{len(ALL_STATES)} states, years {sorted(overall['year'].unique())}")

Loaded 6,966 rows (0 exact duplicates removed)
26 states, years [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


## 3. Analysis functions (one per question)

In [29]:
BAND_ORDER = ["1-10", "11-50", "51-100", ">100", "Not ranked"]


def band_of(rank):
    if pd.isna(rank):
        return "Not ranked"
    if rank <= 10:
        return "1-10"
    if rank <= 50:
        return "11-50"
    if rank <= 100:
        return "51-100"
    return ">100"


def q23_state_change(start=START_YEAR, end=END_YEAR, top_n=TOP_N):
    """Q2.3: change in number of top-N institutions per state between two years."""
    top = overall[overall["rank"] <= top_n]
    counts = top.groupby(["state", "year"]).size().unstack("year", fill_value=0)
    counts = counts.reindex(index=ALL_STATES, columns=[start, end], fill_value=0)
    counts.columns = ["start", "end"]
    counts["change"] = counts["end"] - counts["start"]
    counts.index.name = "state"
    return counts.sort_values(["change", "end"], ascending=False).reset_index()


def q24_compare(states=COMPARE_STATES, top_n=TOP_N):
    """Q2.4: per-year summary for two states."""
    d = overall[overall["state"].isin(states)]
    return (
        d.groupby(["year", "state"])
        .agg(
            ranked=("institution_id", "nunique"),
            top_n_count=("rank", lambda r: int((r <= top_n).sum())),
            mean_rank=("rank", "mean"),
            median_rank=("rank", "median"),
            best_rank=("rank", "min"),
        )
        .reset_index()
    )


def q53_teaching_perception(year=TLR_PR_YEAR):
    """Q5.3: institutions in the top quartile for TLR and bottom quartile for PR."""
    d = overall[overall["year"] == year].dropna(subset=["TLR", "PR"]).copy()
    hi_thr = d["TLR"].quantile(0.75)
    lo_thr = d["PR"].quantile(0.25)
    d["group"] = np.where((d["TLR"] >= hi_thr) & (d["PR"] <= lo_thr),
                          "High TLR, low PR", "Other institutions")
    return d, hi_thr, lo_thr


def q71_transitions(y0=START_YEAR, y1=END_YEAR):
    """Q7.1: where the institutions ranked 51-100 in y0 ended up in y1 (matched on Unified_ID)."""
    a = overall[overall["year"] == y0][["institution_id", "display_name", "rank"]]
    a = a.rename(columns={"rank": "rank_from"})
    b = overall[overall["year"] == y1][["institution_id", "rank"]].rename(columns={"rank": "rank_to"})
    m = a.merge(b, on="institution_id", how="left")   # only institutions present in y0
    m["band_from"] = m["rank_from"].map(band_of)
    m["band_to"] = m["rank_to"].map(band_of)
    base = m[m["rank_from"].between(51, 100)]
    moved = base[base["rank_to"] <= 50]
    stayed = base[base["rank_to"].between(51, 100)]
    fell = base[(base["rank_to"] > 100) | base["rank_to"].isna()]
    matrix = (pd.crosstab(m["band_from"], m["band_to"])
              .reindex(index=BAND_ORDER, columns=BAND_ORDER, fill_value=0))
    return moved, stayed, fell, matrix


def save_figure(fig, name, width, height):
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.write_html(OUT_DIR / f"{name}.html", include_plotlyjs=True)
    try:
        fig.write_image(OUT_DIR / f"{name}.png", width=width, height=height, scale=2)
        print("Saved", OUT_DIR / f"{name}.png")
    except Exception as exc:
        print(f"PNG export failed ({exc}). Open {name}.html in a browser and screenshot it.")


def save_table(table, name):
    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    table.to_csv(TABLE_DIR / f"{name}.csv", index=False)

## 4. India boundaries (needed for the Q2.3 map)

In [30]:
STATE_ALIASES = {
    "Orissa": "Odisha",
    "Uttaranchal": "Uttarakhand",
    "Pondicherry": "Puducherry",
    "NCT of Delhi": "Delhi",
    "Jammu & Kashmir": "Jammu and Kashmir",
    "Andaman & Nicobar Island": "Andaman and Nicobar Islands",
    "Andaman & Nicobar": "Andaman and Nicobar Islands",
}


def canonical_state(name):
    name = str(name).strip()
    return STATE_ALIASES.get(name, name)


def load_india_geojson():
    """Download (once) and load India state boundaries. Returns None if unavailable."""
    if not GEOJSON_PATH.exists():
        GEOJSON_PATH.parent.mkdir(parents=True, exist_ok=True)
        for url in GEOJSON_URLS:
            try:
                with urllib.request.urlopen(url, timeout=20) as resp:
                    GEOJSON_PATH.write_bytes(resp.read())
                print("Downloaded boundaries from", url)
                break
            except Exception as exc:
                print("Could not download", url, "-", exc)
    if not GEOJSON_PATH.exists():
        print("No boundary file. Place an India state GeoJSON at", GEOJSON_PATH)
        return None

    gj = json.loads(GEOJSON_PATH.read_text(encoding="utf-8"))
    features = gj["features"]
    # The state-name property differs between GeoJSON files, so find the one that matches
    best_key, best_hits = None, -1
    for key in features[0]["properties"]:
        hits = sum(canonical_state(f["properties"].get(key, "")) in ALL_STATES for f in features)
        if hits > best_hits:
            best_key, best_hits = key, hits
    for f in features:
        f["properties"]["state_canon"] = canonical_state(f["properties"].get(best_key, ""))

    matched = {f["properties"]["state_canon"] for f in features} & set(ALL_STATES)
    print(f"Name property '{best_key}': matched {len(matched)} of {len(ALL_STATES)} states")
    missing = sorted(set(ALL_STATES) - matched)
    if missing:
        print("States in the data with no boundary (add an alias if needed):", missing)
    return gj


GEOJSON = load_india_geojson()

Name property 'NAME_1': matched 25 of 26 states
States in the data with no boundary (add an alias if needed): ['Telangana']


## 5. Q2.3 · Which states improved their representation in the top 100?

In [31]:
q23 = q23_state_change()
gained = q23[q23["change"] > 0]
lost = q23[q23["change"] < 0]
print(f"{len(gained)} states gained institutions in the top {TOP_N}; {len(lost)} lost.")
print(q23[q23["change"] != 0].to_string(index=False))
save_table(q23, "2.3_state_top100_change")


def fig_q23(table, gj):
    shown = table[table["change"] != 0].sort_values("change")
    bar = go.Bar(
        y=shown["state"], x=shown["change"], orientation="h",
        marker_color=[PAL["gain"] if c > 0 else PAL["loss"] for c in shown["change"]],
        text=[f"{c:+.0f}" for c in shown["change"]], textposition="outside",
        customdata=shown[["start", "end"]].values,
        hovertemplate="<b>%{y}</b><br>%{customdata[0]} → %{customdata[1]} institutions"
                      "<br>Change: %{x:+.0f}<extra></extra>",
        showlegend=False,
    )
    if gj is None:
        fig = go.Figure([bar])
    else:
        limit = max(1, int(table["change"].abs().max()))
        fig = make_subplots(rows=1, cols=2, column_widths=[0.58, 0.42],
                            specs=[[{"type": "geo"}, {"type": "xy"}]],
                            subplot_titles=("Change by state", "States that gained or lost"))
        fig.add_trace(go.Choropleth(
            geojson=gj, locations=table["state"], z=table["change"],
            featureidkey="properties.state_canon", colorscale=CHANGE_SCALE,
            zmid=0, zmin=-limit, zmax=limit,
            marker_line_color="#555", marker_line_width=0.4,
            customdata=table[["start", "end"]].values,
            hovertemplate="<b>%{location}</b><br>%{customdata[0]} → %{customdata[1]} institutions"
                          "<br>Change: %{z:+.0f}<extra></extra>",
            colorbar=dict(title="Change in<br>institutions", x=0.5, len=0.75),
        ), row=1, col=1)
        fig.update_geos(fitbounds="locations", visible=False, bgcolor="#111111",
                        landcolor="#2b2b2b")
        fig.add_trace(bar, row=1, col=2)

    fig.update_layout(
        template=TEMPLATE, width=1300, height=600,
        title=f"Q2.3 · States that improved their representation in the NIRF top {TOP_N}, "
              f"{START_YEAR}–{END_YEAR} ({CATEGORY})",
        xaxis_title="Change in institutions in top 100",
        yaxis_title="State",
        margin=dict(l=40, r=40, t=90, b=40),
    )
    return fig


fig23 = fig_q23(q23, GEOJSON)
# fig23.show()
save_figure(fig23, "2.3_state_top100_change", 1300, 600)

9 states gained institutions in the top 100; 8 lost.
            state  start  end  change
Jammu and Kashmir      0    3       3
      Maharashtra      9   11       2
    Uttar Pradesh      7    9       2
            Delhi      6    8       2
      Uttarakhand      2    4       2
           Punjab      5    6       1
           Odisha      4    5       1
        Rajasthan      3    4       1
       Chandigarh      1    2       1
 Himachal Pradesh      2    1      -1
          Haryana      1    0      -1
          Mizoram      1    0      -1
        Karnataka      8    6      -2
          Gujarat      3    1      -2
        Meghalaya      2    0      -2
       Tamil Nadu     20   17      -3
      West Bengal      7    4      -3
Saved /Users/anubhav/Downloads/DHV_PROJECT/DHV_Final_Project/Screenshots/2.3_state_top100_change.png


## 6. Q2.4 · How do Karnataka and Tamil Nadu compare?

In [32]:
q24 = q24_compare()
save_table(q24, "2.4_state_comparison")
latest = q24[q24["year"] == END_YEAR]
print(f"In {END_YEAR}:")
print(latest[["state", "ranked", "top_n_count", "mean_rank", "best_rank"]].to_string(index=False))


def fig_q24(table, rows):
    colors = {COMPARE_STATES[0]: PAL["state_a"], COMPARE_STATES[1]: PAL["state_b"]}
    fig = make_subplots(rows=1, cols=3, subplot_titles=(
        f"Institutions in top {TOP_N}", "Mean rank per year (lower = better)",
        "Spread of individual ranks, all years"))
    for state in COMPARE_STATES:
        t = table[table["state"] == state]
        c = colors[state]
        fig.add_trace(go.Bar(name=state, x=t["year"], y=t["top_n_count"], marker_color=c,
                             legendgroup=state), row=1, col=1)
        fig.add_trace(go.Scatter(name=state, x=t["year"], y=t["mean_rank"], mode="lines+markers",
                                 line=dict(color=c), marker=dict(color=c), legendgroup=state,
                                 showlegend=False,
                                 hovertemplate="%{x}: mean rank %{y:.1f}<extra>" + state + "</extra>"),
                      row=1, col=2)
        r = rows[rows["state"] == state]
        fig.add_trace(go.Box(name=state, y=r["rank"], marker_color=c, line_color=c,
                             boxpoints="all", jitter=0.4, pointpos=0, legendgroup=state,
                             showlegend=False), row=1, col=3)
    fig.update_yaxes(autorange="reversed", row=1, col=2)
    fig.update_yaxes(autorange="reversed", title_text="Rank (lower = better)", row=1, col=3)
    fig.update_layout(
        template=TEMPLATE, width=1300, height=520, barmode="group",
        title=f"Q2.4 · Karnataka vs Tamil Nadu in the NIRF {CATEGORY} ranking, "
              f"{START_YEAR}–{END_YEAR}",
        margin=dict(l=40, r=40, t=100, b=40),
    )
    fig.update_xaxes(dtick=1, row=1, col=1)
    fig.update_xaxes(dtick=1, row=1, col=2)
    return fig


ranks_kt = overall[overall["state"].isin(COMPARE_STATES) & overall["rank"].notna()]
fig24 = fig_q24(q24, ranks_kt)
# fig24.show()
save_figure(fig24, "2.4_karnataka_vs_tamilnadu", 1300, 520)

In 2025:
     state  ranked  top_n_count  mean_rank  best_rank
 Karnataka       6            6  44.000000        2.0
Tamil Nadu      17           17  51.352941        1.0
Saved /Users/anubhav/Downloads/DHV_PROJECT/DHV_Final_Project/Screenshots/2.4_karnataka_vs_tamilnadu.png


## 7. Q5.3 · Which institutions have high teaching but low perception?

In [33]:
q53_df, hi_thr, lo_thr = q53_teaching_perception()
qualifiers = q53_df[q53_df["group"] == "High TLR, low PR"].sort_values("TLR", ascending=False)
print(f"{TLR_PR_YEAR}: high TLR threshold (75th percentile) = {hi_thr:.2f}; "
      f"low PR threshold (25th percentile) = {lo_thr:.2f}")
print(f"{len(qualifiers)} institutions qualify.")
print(qualifiers[["display_name", "state", "TLR", "PR"]].to_string(index=False))
save_table(qualifiers[["display_name", "state", "TLR", "PR"]], "5.3_teaching_vs_perception")


def fig_q53(d, hi, lo, year):
    others = d[d["group"] == "Other institutions"]
    hits = d[d["group"] == "High TLR, low PR"]
    x_min, x_max = d["PR"].min() - 2, d["PR"].max() + 2
    y_min, y_max = d["TLR"].min() - 2, d["TLR"].max() + 2
    fig = go.Figure()
    fig.add_shape(type="rect", x0=x_min, x1=lo, y0=hi, y1=y_max,
                  fillcolor="rgba(213,94,0,0.12)", line_width=0, layer="below")
    fig.add_trace(go.Scatter(
        x=others["PR"], y=others["TLR"], mode="markers", name="Other institutions",
        marker=dict(color=PAL["neutral"], size=8, opacity=0.6),
        customdata=others["display_name"],
        hovertemplate="<b>%{customdata}</b><br>PR %{x:.1f} · TLR %{y:.1f}<extra></extra>"))
    fig.add_trace(go.Scatter(
        x=hits["PR"], y=hits["TLR"], mode="markers+text", name="High TLR, low PR",
        marker=dict(color=PAL["loss"], size=13, line=dict(color="white", width=0.8)),
        text=hits["display_name"].str.slice(0, 22), textposition="top center",
        customdata=hits["display_name"],
        hovertemplate="<b>%{customdata}</b><br>PR %{x:.1f} · TLR %{y:.1f}<extra></extra>"))
    fig.add_hline(y=hi, line_dash="dash", line_color="#cccccc",
                  annotation_text=f"Top quartile TLR ≥ {hi:.1f}", annotation_position="top left")
    fig.add_vline(x=lo, line_dash="dash", line_color="#cccccc",
                  annotation_text=f"Bottom quartile PR ≤ {lo:.1f}", annotation_position="top right")
    fig.update_layout(
        template=TEMPLATE, width=1000, height=640,
        title=f"Q5.3 · High teaching but low perception, {year} ({CATEGORY})",
        xaxis_title="Perception (PR) score", yaxis_title="Teaching, learning & resources (TLR) score",
        xaxis=dict(range=[x_min, x_max]), yaxis=dict(range=[y_min, y_max]),
        margin=dict(l=40, r=40, t=90, b=40),
    )
    return fig


fig53 = fig_q53(q53_df, hi_thr, lo_thr, TLR_PR_YEAR)
# fig53.show()
save_figure(fig53, "5.3_teaching_vs_perception", 1000, 640)

2025: high TLR threshold (75th percentile) = 73.76; low PR threshold (25th percentile) = 10.02
4 institutions qualify.
                                        display_name             state   TLR   PR
Saveetha Institute of Medical and Technical Sciences        Tamil Nadu 76.12 6.42
                    King George's Medical University     Uttar Pradesh 75.27 7.11
                               University of Kashmir Jammu and Kashmir 74.05 6.07
          Kalinga Institute of Industrial Technology            Odisha 74.01 8.46
Saved /Users/anubhav/Downloads/DHV_PROJECT/DHV_Final_Project/Screenshots/5.3_teaching_vs_perception.png


In [34]:
def fig_q53(d, hi, lo, year):
    others = d[d["group"] == "Other institutions"]
    hits = (d[d["group"] == "High TLR, low PR"]
            .sort_values("TLR", ascending=False)
            .reset_index(drop=True))
    hits["No."] = range(1, len(hits) + 1)

    x_min, x_max = d["PR"].min() - 2, d["PR"].max() + 2
    y_min, y_max = d["TLR"].min() - 2, d["TLR"].max() + 2

    fig = make_subplots(
        rows=1, cols=2, column_widths=[0.6, 0.4],
        specs=[[{"type": "xy"}, {"type": "domain"}]],
        subplot_titles=("Scatter (numbers match the list)", "Qualifying institutions"),
    )

    # Shaded quadrant: high TLR and low PR
    fig.add_shape(type="rect", xref="x", yref="y",
                  x0=x_min, x1=lo, y0=hi, y1=y_max,
                  fillcolor="rgba(213,94,0,0.12)", line_width=0, layer="below")

    # Threshold lines
    fig.add_shape(type="line", xref="x", yref="y",
                  x0=x_min, x1=x_max, y0=hi, y1=hi,
                  line=dict(dash="dash", color="#cccccc"))
    fig.add_shape(type="line", xref="x", yref="y",
                  x0=lo, x1=lo, y0=y_min, y1=y_max,
                  line=dict(dash="dash", color="#cccccc"))
    fig.add_annotation(xref="x", yref="y", x=x_max, y=hi, text=f"Top quartile TLR ≥ {hi:.1f}",
                       showarrow=False, xanchor="right", yanchor="bottom", font=dict(color="#cccccc"))
    fig.add_annotation(xref="x", yref="y", x=lo, y=y_max, text=f"Bottom quartile PR ≤ {lo:.1f}",
                       showarrow=False, xanchor="left", yanchor="top", font=dict(color="#cccccc"))

    # Other institutions (grey)
    fig.add_trace(go.Scatter(
        x=others["PR"], y=others["TLR"], mode="markers", name="Other institutions",
        marker=dict(color=PAL["neutral"], size=8, opacity=0.6),
        customdata=others["display_name"],
        hovertemplate="<b>%{customdata}</b><br>PR %{x:.1f} · TLR %{y:.1f}<extra></extra>",
    ), row=1, col=1)

    # Qualifiers (numbered markers)
    fig.add_trace(go.Scatter(
        x=hits["PR"], y=hits["TLR"], mode="markers+text", name="High TLR, low PR",
        marker=dict(color=PAL["loss"], size=20, line=dict(color="white", width=1)),
        text=hits["No."].astype(str), textposition="middle center",
        textfont=dict(color="white", size=11),
        customdata=hits["display_name"],
        hovertemplate="<b>%{customdata}</b><br>PR %{x:.1f} · TLR %{y:.1f}<extra></extra>",
    ), row=1, col=1)

    # Names table (right panel)
    fig.add_trace(go.Table(
        header=dict(values=["No.", "Institution", "State", "TLR", "PR"],
                    fill_color="#262626", font=dict(color="white", size=12), align="left"),
        cells=dict(values=[hits["No."], hits["display_name"], hits["state"],
                           hits["TLR"].round(2), hits["PR"].round(2)],
                   fill_color="#111111", font=dict(color="white", size=11), align="left",
                   height=32),
    ), row=1, col=2)

    fig.update_xaxes(title_text="Perception (PR) score", range=[x_min, x_max], row=1, col=1)
    fig.update_yaxes(title_text="Teaching, learning & resources (TLR) score",
                     range=[y_min, y_max], row=1, col=1)
    fig.update_layout(
        template=TEMPLATE, width=1400, height=560,
        title=f"Q5.3 · High teaching but low perception, {year} ({CATEGORY})",
        margin=dict(l=40, r=40, t=100, b=40),
    )
    return fig


fig53 = fig_q53(q53_df, hi_thr, lo_thr, TLR_PR_YEAR)
# fig53.show()
save_figure(fig53, "5.3_teaching_vs_perception", 1400, 560)

Saved /Users/anubhav/Downloads/DHV_PROJECT/DHV_Final_Project/Screenshots/5.3_teaching_vs_perception.png


## 8. Q7.1 · How many institutions moved from 51–100 into the top 50?

In [35]:
moved, stayed, fell, matrix = q71_transitions()
summary = pd.DataFrame({
    "outcome": ["Moved into top 50", "Stayed in 51–100", "Fell below 100 or not ranked"],
    "institutions": [len(moved), len(stayed), len(fell)],
})
print(f"{len(moved)} institutions moved from the 51–100 band into the top 50 "
      f"({START_YEAR} → {END_YEAR}). Starting band size: {len(moved) + len(stayed) + len(fell)}.")
print(moved[["display_name", "rank_from", "rank_to"]].sort_values("rank_to").to_string(index=False))
print()
print(matrix.to_string())
save_table(moved[["display_name", "rank_from", "rank_to"]], "7.1_moved_into_top50")
summary.to_csv(TABLE_DIR / "7.1_outcome_summary.csv", index=False)


def fig_q71(summary, matrix):
    fig = make_subplots(rows=1, cols=2, column_widths=[0.42, 0.58],
                        subplot_titles=(f"Where the {START_YEAR} band 51–100 went",
                                        "Band-to-band transition counts"))
    fig.add_trace(go.Bar(
        x=summary["outcome"], y=summary["institutions"],
        marker_color=[PAL["gain"], PAL["neutral"], PAL["loss"]],
        text=summary["institutions"], textposition="outside", showlegend=False,
        hovertemplate="%{x}: %{y}<extra></extra>"), row=1, col=1)
    fig.add_trace(go.Heatmap(
        z=matrix.values, x=list(matrix.columns), y=list(matrix.index),
        colorscale=COUNT_SCALE, text=matrix.values, texttemplate="%{text}",
        hovertemplate="From %{y} (" + str(START_YEAR) + ") to %{x} (" + str(END_YEAR)
                      + "): %{z}<extra></extra>",
        colorbar=dict(title="Count", x=1.02)), row=1, col=2)
    fig.update_yaxes(title_text="Institutions", row=1, col=1)
    fig.update_yaxes(title_text=f"Band in {START_YEAR}", autorange="reversed", row=1, col=2)
    fig.update_xaxes(title_text=f"Band in {END_YEAR}", row=1, col=2)
    fig.update_layout(
        template=TEMPLATE, width=1300, height=540,
        title=f"Q7.1 · Ranking-band transitions {START_YEAR} → {END_YEAR} ({CATEGORY}): "
              f"{int(summary['institutions'].iloc[0])} moved from 51–100 into the top 50",
        margin=dict(l=40, r=40, t=100, b=40), showlegend=False)
    return fig


fig71 = fig_q71(summary, matrix)
# fig71.show()
save_figure(fig71, "7.1_band_transitions", 1300, 540)

12 institutions moved from the 51–100 band into the top 50 (2017 → 2025). Starting band size: 50.
                                                            display_name  rank_from  rank_to
                              S.R.M. Institute of Science and Technology       55.0     22.0
                    Saveetha Institute of Medical and Technical Sciences       91.0     23.0
                              Kalinga Institute of Industrial Technology       79.0     27.0
      Indian Institute of Technology (Banaras Hindu University) Varanasi       70.0     31.0
                 Indian Institute of Technology (Indian School of Mines)       53.0     35.0
                                    Indian Institute of Technology Patna       83.0     36.0
                                    Amity University (Gautam Budh Nagar)       86.0     37.0
                            JSS Academy of Higher Education and Research       74.0     38.0
                              Indian Institute of Technology Gand

## 9. Run check

In [36]:
print("Figures and tables written to:", OUT_DIR)
for p in sorted(OUT_DIR.glob("*")):
    print("  ", p.relative_to(ROOT))

Figures and tables written to: /Users/anubhav/Downloads/DHV_PROJECT/DHV_Final_Project/Screenshots
   Screenshots/2.3_state_top100_change.html
   Screenshots/2.3_state_top100_change.png
   Screenshots/2.4_karnataka_vs_tamilnadu.html
   Screenshots/2.4_karnataka_vs_tamilnadu.png
   Screenshots/5.3_teaching_vs_perception.html
   Screenshots/5.3_teaching_vs_perception.png
   Screenshots/7.1_band_transitions.html
   Screenshots/7.1_band_transitions.png
   Screenshots/tables
